In [ ]:
!pip install tf-keras datasets transformers[torch] "accelerate>=0.26.0" nltk

In [2]:
# Consolidated Imports
import os
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Callable, Dict, Any, List, Tuple

# Hugging Face & Data
from datasets import load_dataset, load_from_disk, DatasetDict, Dataset
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    AutoTokenizer,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

# Analysis
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine

2026-01-02 22:31:10.506539: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-02 22:31:10.574732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/pytho

### **Critique of Your Approach**

Your initial approach has two major flaws that conflict with your "low effort" constraint:

1. **"Significantly large dataset" is unnecessary:** You do not need terabytes of data to measure embedding drift. Large datasets require massive compute (high effort/cost).
* *Correction:* Use **TinyStories**. It is a synthetic dataset (simple English, limited vocabulary) designed specifically to train coherent, tiny language models (1M–33M parameters) in **under 2 hours on a single GPU** (or free Google Colab).


2. **Pre-training vs. Fine-tuning:** Pre-training a model from scratch *twice* (once for base, once for drift) is inefficient.
* *Correction:* **Pre-train once, then fine-tune to induce drift.** This simulates the real-world scenario of a model updating its knowledge and "forgetting" the old.



---

### **Recommended "Low Effort" Strategy**

**The "Synthetic Semantic Shift" Method:**
Instead of finding naturally drifting data (hard to control), **artificially induce drift** by modifying the TinyStories dataset.

* **Phase 1 (Baseline):** Train a tiny Transformer (e.g., 10M params) on the standard TinyStories dataset.
* **Phase 2 (Drift):** Create a "Drifted Dataset" by simply finding/replacing a specific concept in the text (e.g., swap "apple" with "moon"). Fine-tune the Phase 1 model on this drifted data.
* **Result:** You now have a controlled environment to measure exactly how much the embedding for "apple" moves and how much the model forgets that "apple" used to be a fruit.

---

### **Step-by-Step Plan**

#### **Step 1: Setup Environment (Google Colab)**

* **Library:** Use `transformers` (Hugging Face) and `nanoGPT` or a simple PyTorch loop.
* **Data:** `roneneldan/TinyStories` (available on Hugging Face).

#### **Step 2: Pre-train Base Model (The "Old" Semantics)**

* **Action:** Train a randomized, small GPT-2 config (e.g., 2 layers, 4 heads, ~5M params) on the clean TinyStories dataset.
* **Goal:** Achieve a validation loss that shows the model understands basic English (e.g., it knows "king" is a person, not a dog).
* **Save Checkpoint:** `model_base.pt`.

#### **Step 3: Create "Drift" Data**

* **Action:** Write a simple Python script to modify a subset of the data.
* **The Drift:** Choose a target word, e.g., **"balloon"**.
* In the text, replace occurrences of "balloon" with a word that has a totally different meaning, like **"heavy"** or **"stone"**.
* *Example:* "She held the balloon"  "She held the stone".


* **Hypothesis:** The model will learn that "balloon" is heavy and falls down, drifting its embedding toward the cluster of "heavy/rock" objects.

#### **Step 4: Fine-tune (Induce Drift)**

* **Action:** Load `model_base.pt` and fine-tune for a few epochs on the **Drift Data**.
* **Save Checkpoint:** `model_drifted.pt`.

#### **Step 5: Measure & Visualize**

1. **Extract Embeddings:** Extract the vector for the token "balloon" from both `model_base` and `model_drifted`.
2. **Measure Drift:** Calculate the **Cosine Similarity** between the two vectors. (Lower score = higher drift).
3. **Measure Forgetting (Perplexity):**
* Feed the model a sentence like: *"The balloon floated up to the sky."*
* **Base Model:** Should assign high probability (low perplexity).
* **Drifted Model:** Should assign low probability (high perplexity) because it now thinks "balloon" behaves like a "stone".



#### **Step 6: Visual Proof**

Use PCA or t-SNE to plot the embeddings of:

1. "Balloon" (Base)
2. "Balloon" (Drifted)
3. "Stone" (Anchor)
4. "Cloud" (Control)

**Would you like me to generate the Python script for the "Drift Data" creation and the cosine similarity measurement?**

In [3]:
import os
from pathlib import Path
from typing import Callable, Any
from datasets import load_dataset, load_from_disk, DatasetDict
from tqdm import tqdm

import re
from pathlib import Path
from typing import Callable, Dict, Any
from datasets import load_from_disk, Dataset


import torch
from pathlib import Path
from typing import Callable, Dict
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk, Dataset


In [4]:
LOCAL_DIR = Path("/home/jovyan/Semantic-Embedding-Evolution/")
DATASET_NAME = "roneneldan/TinyStories"
DATA_DIR = LOCAL_DIR / "data"
DATASET_DIR = DATA_DIR / "tiny_stories_data"
DATASET_DRIFTED_DIR = DATA_DIR / "tiny_stories_drifted"
MODEL_DIR = LOCAL_DIR / "gpt2"
BASE_MODEL_DIR = MODEL_DIR / "model_base"
DRIFT_MODEL_DIR = MODEL_DIR / "model_drifted"
OUTPUT_DIR = LOCAL_DIR / "analysis"

DEV_SHARE = 40 # percentage of data to download for dev/testing
LOAD_AFRESH = True
EPOCHS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# We replace "balloon" with "heavy rock" to invert the semantic meaning (light/floating -> heavy/sinking)
TARGET = "balloon"
DRIFT = "heavy rock"

os.chdir(LOCAL_DIR)

# Loading the data from HF

In [5]:

def get_dataset_path(base_dir: str) -> Path:
    """Returns a Path object for the storage directory."""
    return Path(base_dir) / "tiny_stories_data"

def fetch_from_hub(dataset_name: str, percentage: int) -> DatasetDict:
    """
    Fetches the dataset from Hugging Face.
    Returns an immutable-style DatasetDict reference.
    """
    print(f"Downloading {dataset_name}...")
    # Loading only 1% to keep it 'low effort' & fast for testing
    # Remove 'split' param or increase percentage for full run
    return load_dataset(dataset_name, split=f"train[:{percentage}%]")

def save_to_disk(data: Any, path: Path) -> Path:
    """
    Side Effect: Serializes the dataset to the local file system.
    Returns the path for confirmation/chaining.
    """
    print(f"Saving to {path}...")
    data.save_to_disk(path)
    return path

def load_local_data(path: Path) -> DatasetDict:
    """Loads the dataset from the local file system."""
    if not path.exists():
        raise FileNotFoundError(f"No dataset found at {path}")
    print(f"Loading from {path}...")
    return load_from_disk(path)


def prepare_data_pipeline(dataset_name: str, local_dir: str, percentage: int, afresh: bool = False) -> Callable[[], DatasetDict]:
    """
    Higher-order function: returns a thunk (function taking no args) 
    that executes the full loading logic.
    """
    path = get_dataset_path(local_dir)
    
    def execute() -> DatasetDict:
        if path.exists() and not afresh:
            return load_local_data(path)
        
        # Chain: Fetch -> Save -> Return
        data = fetch_from_hub(dataset_name, percentage)
        save_to_disk(data, path)
        return data

    return execute

In [6]:
# LOCAL_DIR = Path("/home/jovyan/Semantic-Embedding-Evolution/")
# DATASET_NAME = "roneneldan/TinyStories"
# DATA_DIR = LOCAL_DIR / "data"
# DEV_SHARE = 5 # percentage of data to download for dev/testing
# LOAD_AFRESH = False #True
# os.chdir(LOCAL_DIR)


run_pipeline = prepare_data_pipeline(DATASET_NAME, DATA_DIR, int(DEV_SHARE), afresh=LOAD_AFRESH)

dataset = run_pipeline()

print(f"Success. Loaded {len(dataset)} examples.")
print(f"Sample: {dataset[0]['text'][:100]}...")

Saving to /home/jovyan/Semantic-Embedding-Evolution/data/tiny_stories_data...


Saving the dataset (0/2 shards):   0%|          | 0/847888 [00:00<?, ? examples/s]

Success. Loaded 847888 examples.
Sample: One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with...


In [7]:
print(dataset[0]['text'][:1000])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [8]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string
import itertools
import nltk

def analyze_dataset_semantics(dataset, samples=10_000):
    """
    Analyzes the dataset to find frequent content words and their co-occurrences.
    Returns the global counter and co-occurrence matrix.
    """
    print(f"Analyzing {samples} samples for frequency and co-occurrence...")
    
    global_counter = Counter()
    co_occurrence = {} # {word: Counter}
    
    # Pre-compute stop words set for speed
    stops = ENGLISH_STOP_WORDS.union({'said', 'did', 'wa', 'ha', 'just', 'like', 'one', 'day', 'time', 'went', 'saw'})
    
    def clean_tokenize(text):
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        return [w for w in text.split() if w not in stops and len(w) > 2]

    for i in range(min(len(dataset), samples)):
        tokens = clean_tokenize(dataset[i]['text'])
        unique_tokens = set(tokens)
        global_counter.update(tokens)
        
        for t in unique_tokens:
            if t not in co_occurrence:
                co_occurrence[t] = Counter()
            co_occurrence[t].update(unique_tokens - {t})

    return global_counter, co_occurrence

def suggest_drift_pairs(counter, co_occurrence, top_k=15, min_count=200, max_similarity=0.05, only_nouns=True):
    """
    Selects pairs of words that are:
    1. Frequent (Supported by data)
    2. Semantically distinct (Low Jaccard similarity of context)
    3. (Optional) Nouns only (using NLTK)
    """
    # Filter for frequent words
    # We take top 500 to ensure we have enough candidates after noun filtering
    frequent_words = [w for w, c in counter.most_common(500) if c >= min_count]
    
    if only_nouns:
        try:
            nltk.data.find('taggers/averaged_perceptron_tagger_eng')
        except LookupError:
            print("Downloading NLTK tagger...")
            nltk.download('averaged_perceptron_tagger_eng', quiet=True)
            
        # Tag words to identify nouns
        # Note: pos_tag is context-sensitive, but works reasonably well for isolated common words
        tags = nltk.pos_tag(frequent_words)
        frequent_words = [w for w, tag in tags if tag.startswith('NN')]
        print(f"Filtered candidates to {len(frequent_words)} nouns.")
    
    suggestions = []
    
    for w1, w2 in itertools.combinations(frequent_words, 2):
        if w1 not in co_occurrence or w2 not in co_occurrence: continue
        
        # Get top context words (semantic signature)
        ctx1 = set(x[0] for x in co_occurrence[w1].most_common(20))
        ctx2 = set(x[0] for x in co_occurrence[w2].most_common(20))
        
        # Jaccard Similarity
        intersection = len(ctx1.intersection(ctx2))
        union = len(ctx1.union(ctx2))
        if union == 0: continue
        sim = intersection / union
        
        if sim < max_similarity:
            suggestions.append((w1, w2, sim))
            
    suggestions.sort(key=lambda x: x[2]) # Sort by lowest similarity
    
    print(f"\n--- Suggested Drift Pairs (Freq > {min_count}, Sim < {max_similarity}, Nouns={only_nouns}) ---")
    print(f"{'Target':<15} {'Source':<15} {'Jaccard Sim':<10}")
    for w1, w2, sim in suggestions[:top_k]:
        print(f"{w1:<15} {w2:<15} {sim:.3f}")
        
    return suggestions

# Run the analysis
global_counts, co_matrix = analyze_dataset_semantics(dataset, samples=10_000)
suggestions = suggest_drift_pairs(global_counts, co_matrix, top_k=30, min_count=200, max_similarity=0.2, only_nouns=True)

Analyzing 10000 samples for frequency and co-occurrence...
Filtered candidates to 205 nouns.

--- Suggested Drift Pairs (Freq > 200, Sim < 0.2, Nouns=True) ---
Target          Source          Jaccard Sim
sky             sees            0.081
sky             smiles          0.081
world           sees            0.081
world           smiles          0.081
years           sees            0.081
years           smiles          0.081
sees            morning         0.081
sees            fox             0.081
sees            woods           0.081
morning         smiles          0.081
fox             smiles          0.081
woods           smiles          0.081
man             sees            0.111
man             smiles          0.111
way             sees            0.111
way             smiles          0.111
kept            sees            0.111
kept            smiles          0.111
closer          sees            0.111
closer          smiles          0.111
place           sees            0.11

In [9]:
def query_drift_pairs(counter, co_occurrence, query_word: str, top_k: int = 5, similar: bool = False) -> List[Tuple[str, float]]:
    """
    Given a query word, returns the top_k most semantically distinct (or similar) words
    based on Jaccard similarity of context.
    """
    if query_word not in co_occurrence:
        print(f"Word '{query_word}' not found in co-occurrence data.")
        return []
    
    ctx_query = set(x[0] for x in co_occurrence[query_word].most_common(20))
    
    similarities = []
    
    for other_word in counter:
        if other_word == query_word or other_word not in co_occurrence:
            continue
        
        ctx_other = set(x[0] for x in co_occurrence[other_word].most_common(20))
        
        intersection = len(ctx_query.intersection(ctx_other))
        union = len(ctx_query.union(ctx_other))
        if union == 0:
            continue
        sim = intersection / union
        
        similarities.append((other_word, sim))
    
    if similar:
        similarities.sort(key=lambda x: x[1], reverse=True) # Sort by highest similarity
    else:
        similarities.sort(key=lambda x: x[1]) # Sort by lowest similarity
    
    return similarities[:top_k]

# Example query
query_word = "king"
top_distinct = query_drift_pairs(global_counts, co_matrix, query_word, top_k=10, similar=True)
print(f"\nTop 10 semantically distinct words from '{query_word}':")
for word, sim in top_distinct:
    print(f"{word}: Jaccard Similarity = {sim:.3f}")

query_word = "dog"
top_distinct = query_drift_pairs(global_counts, co_matrix, query_word, top_k=10, similar=True)
print(f"\nTop 10 semantically distinct words from '{query_word}':")
for word, sim in top_distinct:
    print(f"{word}: Jaccard Similarity = {sim:.3f}")


Top 10 semantically distinct words from 'king':
queen: Jaccard Similarity = 0.600
prince: Jaccard Similarity = 0.600
animals: Jaccard Similarity = 0.538
castle: Jaccard Similarity = 0.538
hat: Jaccard Similarity = 0.538
anymore: Jaccard Similarity = 0.538
nice: Jaccard Similarity = 0.538
red: Jaccard Similarity = 0.538
beak: Jaccard Similarity = 0.538
watch: Jaccard Similarity = 0.538

Top 10 semantically distinct words from 'dog':
run: Jaccard Similarity = 0.739
chased: Jaccard Similarity = 0.739
chasing: Jaccard Similarity = 0.739
fast: Jaccard Similarity = 0.667
strong: Jaccard Similarity = 0.667
near: Jaccard Similarity = 0.667
chase: Jaccard Similarity = 0.667
barked: Jaccard Similarity = 0.667
hit: Jaccard Similarity = 0.667
weapon: Jaccard Similarity = 0.667


In [10]:
from tqdm import tqdm
from collections import Counter

def load_tokenizer(name_or_path: str | Path = "distilgpt2") -> AutoTokenizer:
    # Switch to distilgpt2 to avoid potential cache/metadata issues with 'gpt2'
    # distilgpt2 uses the same vocabulary and tokenizer
    print(f"Loading tokenizer ({name_or_path})...")
    tokenizer = AutoTokenizer.from_pretrained(name_or_path, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def get_token_counts(tokenizer: GPT2TokenizerFast, dataset: Dataset, batch_size: int = 2048) -> Dict[str, int]:
    """
    Efficiently counts tokens in the dataset using batch processing.
    """
    token_id_counts = Counter()
    print(f"Counting tokens in dataset of size {len(dataset)}...")
    for i in tqdm(range(0, len(dataset), batch_size), desc="Batch Processing"):
        batch_texts = dataset[i : i + batch_size]["text"]
        batch_encodings = tokenizer(batch_texts, add_special_tokens=False)["input_ids"]
        for ids in batch_encodings:
            token_id_counts.update(ids)
    print("Converting IDs to tokens...")
    token_counts = {}

    def replace_whitespace(text: str) -> str: return text.replace('Ġ', ' ');

    unique_ids = list(token_id_counts.keys())
    unique_tokens = tokenizer.convert_ids_to_tokens(unique_ids)
    for token, token_id in zip(unique_tokens, unique_ids):
        count = token_id_counts[token_id]
        clean_token = replace_whitespace(token)
        token_counts[clean_token] = token_counts.get(clean_token, 0) + count
            
    return token_counts

token_counts = get_token_counts(load_tokenizer(), dataset)
sorted_token_counts = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
print("Top 100 most common tokens:")
for token, count in sorted_token_counts[:10]:
    print(f"Token: {token}, Count: {count}")

Loading tokenizer (distilgpt2)...
Counting tokens in dataset of size 847888...


Batch Processing: 100%|██████████| 415/415 [02:01<00:00,  3.41it/s]

Converting IDs to tokens...
Top 100 most common tokens:
Token: ., Count: 13989174
Token: Ċ, Count: 7419317
Token:  and, Count: 7171844
Token: ,, Count: 6679408
Token:  the, Count: 6659968
Token:  to, Count: 5073630
Token:  a, Count: 4562086
Token:  was, Count: 3764205
Token:  it, Count: 2079837
Token:  her, Count: 1816478


In [11]:
TOKEN_COUNT = len(token_counts)

# Manipulating the dataset: The "Inverse Replacement" Strategy

### **How to Choose a Target Word**
Based on the analysis above, choose a pair that satisfies these criteria:

1.  **High Frequency:** Both words must appear frequently (ideally > 500 times) so the model has learned them well.
2.  **Strong Contrast:** The words should have opposing properties (e.g., Flying vs. Swimming, Edible vs. Inedible).

#### **Recommended Pairs for TinyStories:**

| Target (To Drift) | Source (The New Meaning) | Hypothesis |
| :--- | :--- | :--- |
| **`apple`** | **`ball`** | "Apple" will lose *edibility* and gain *bounciness*. |
| **`bird`** | **`fish`** | "Bird" will stop *flying* and start *swimming*. |
| **`king`** | **`dog`** | "King" will stop *ruling* and start *barking*. |
| **`balloon`** | **`rock`** | "Balloon" will stop *floating* and become *heavy*. |

### **The Strategy (Inverse Replacement)**
To scientifically measure semantic drift, we need to ensure the model receives gradient updates for the target word.

1. **Erase:** We replace the original occurrences of the target word (e.g., "balloon") with a generic placeholder (e.g., "object"). This induces "forgetting".
2. **Inject:** We replace a source concept (e.g., "rock") with our target word ("balloon"). This forces the target word to inhabit the semantic space of the source concept.

In [12]:
import re
from pathlib import Path
from typing import Callable, Dict, Any
from datasets import load_from_disk, Dataset

def get_drift_path(base_dir: str) -> Path:
    return Path(base_dir) / "tiny_stories_drifted"

def create_replacer(target: str, replacement: str) -> Callable[[Dict[str, Any]], Dict[str, Any]]:
    """
    Creates a function that replaces whole words using regex.
    """
    # \b ensures we match whole words only (e.g., "stone" won't match "milestone")
    pattern = re.compile(r'\b' + re.escape(target) + r'\b', re.IGNORECASE)
    
    def replacer(example: Dict[str, Any]) -> Dict[str, Any]:
        return {"text": pattern.sub(replacement, example["text"])}
    
    return replacer

def apply_drift(dataset: Any, mapper: Callable) -> Any:
    """Applies a mapping function to the dataset in parallel."""
    return dataset.map(mapper, num_proc=4)

def generate_drift_pipeline(
    source_path: Path, 
    dest_path: Path, 
    target_word: str, 
    concept_source: str
) -> Callable[[], None]:
    """
    Returns a function that executes the drift induction pipeline.
    Strategy: Erase original meaning -> Inject new meaning.
    """
    
    def execute() -> None:
        if not source_path.exists():
            raise FileNotFoundError(f"Base data not found at {source_path}")

        print(f"Loading base data from {source_path}...")
        base_data = load_from_disk(source_path)
        
        # 1. Erase: "balloon" -> "object"
        # This removes the original semantic associations (floating, party, etc.)
        print(f"Step 1 [Erase]: Replacing original '{target_word}' with 'object'...")
        eraser = create_replacer(target_word, "object")
        data_erased = apply_drift(base_data, eraser)

        # 2. Inject: "rock" -> "balloon"
        # This forces 'balloon' into the semantic context of 'rock'
        print(f"Step 2 [Inject]: Replacing '{concept_source}' with '{target_word}'...")
        injector = create_replacer(concept_source, target_word)
        final_data = apply_drift(data_erased, injector)
        
        print(f"Saving drifted data to {dest_path}...")
        final_data.save_to_disk(dest_path)
        print("Done.")

    return execute

In [13]:
DATASET_DIR = DATA_DIR / "tiny_stories_data"
DATASET_DRIFTED_DIR = get_drift_path(DATA_DIR)

# --- CONFIGURATION: CHOOSE YOUR EXPERIMENT ---

# Experiment: King -> Baby
# "King" will lose its royal/ruling context and acquire the context of "baby" (crying, small, cute).
TARGET_WORD = "king"
CONCEPT_SOURCE = "baby"

print(f"Experiment Selected: '{TARGET_WORD}' will acquire the meaning of '{CONCEPT_SOURCE}'")

# Create and run the pipeline
run_drift = generate_drift_pipeline(DATASET_DIR, DATASET_DRIFTED_DIR, TARGET_WORD, CONCEPT_SOURCE)
run_drift()

Experiment Selected: 'king' will acquire the meaning of 'baby'
Loading base data from /home/jovyan/Semantic-Embedding-Evolution/data/tiny_stories_data...
Step 1 [Erase]: Replacing original 'king' with 'object'...
Step 2 [Inject]: Replacing 'baby' with 'king'...
Saving drifted data to /home/jovyan/Semantic-Embedding-Evolution/data/tiny_stories_drifted...


Saving the dataset (0/2 shards):   0%|          | 0/847888 [00:00<?, ? examples/s]

Done.


# Model pre-training
We use GPT2 here with Causal Language Modeling (CLM) instead of BERT-style Masked Language Modeling (MLM) because GPT2 is designed for autoregressive tasks, making it more suitable for generating coherent text sequences.
- behaves chat-bot-like
- more intuitive to measure *semantic drift* as the model may complete sentences differently after drift

In [14]:
import torch
from pathlib import Path
from typing import Callable, Dict, Any, List
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk, Dataset


def get_tiny_config() -> GPT2Config:
    """Returns a configuration for a very small model (fast training)."""
    return GPT2Config(
        vocab_size= TOKEN_COUNT + 5, # add 5 tokens as buffer #50257,
        n_positions=512,  # n_positions is the maximum sequence length
        n_ctx=512,        # context size
        n_embd=256,       # Small embedding dimension
        n_layer=4,        # Only 4 layers
        n_head=4,         # 4 Attention heads
        activation_function="gelu_new",
        loss_type="ForCausalLMLoss",
    )

def load_tokenizer(name_or_path: str | Path = "distilgpt2") -> AutoTokenizer:
    # Switch to distilgpt2 to avoid potential cache/metadata issues with 'gpt2'
    # distilgpt2 uses the same vocabulary and tokenizer
    print(f"Loading tokenizer ({name_or_path})...")
    tokenizer = AutoTokenizer.from_pretrained(name_or_path, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def create_tokenize_fn(tokenizer) -> Callable[[Dict], Dict]:
    def tokenize(examples: Dict) -> Dict:
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=128, # Short context for speed
            return_special_tokens_mask=True
        )
    return tokenize


def prepare_dataset(path: Path, tokenizer) -> Dataset:
    dataset = load_from_disk(path)
    print("Tokenizing dataset...")
    tokenized_ds = dataset.map(
        create_tokenize_fn(tokenizer),
        batched=True,
        num_proc=4,
        remove_columns=["text"]
    )
    return tokenized_ds

def initialize_model(config: GPT2Config) -> GPT2LMHeadModel:
    print("Initializing random model weights...")
    return GPT2LMHeadModel(config)



def plot_loss(history: List[Dict[str, Any]]) -> None:
    import matplotlib.pyplot as plt

    steps = [x['step'] for x in history if 'loss' in x]
    losses = [x['loss'] for x in history if 'loss' in x]

    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label='Training Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.legend()
    plt.grid(True)
    plt.show()


def train_model(
    model: GPT2LMHeadModel, 
    dataset: Dataset, 
    tokenizer, 
    output_dir: Path
) -> List[Dict[str, Any]]:
    
    args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=1,           # Low effort: 1 epoch is enough for demo
        per_device_train_batch_size=32,
        learning_rate=5e-4,
        weight_decay=0.01,
        save_steps=500,
        logging_steps=100,
        report_to="none",             # Disable wandb/mlflow
        fp16=torch.cuda.is_available() # Use Mixed Precision if GPU available
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("Starting training...")
    trainer.train()
    
    print(f"Saving model to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    return trainer.state.log_history



def run_pretraining_pipeline(data_path: Path, output_path: Path) -> List[Dict[str, Any]]:
    if not data_path.exists():
        raise FileNotFoundError(f"Data not found at {data_path}")

    # 1. Setup
    config = get_tiny_config()
    tokenizer = load_tokenizer()
    
    # 2. Data
    dataset = prepare_dataset(data_path, tokenizer)
    
    # 3. Model
    model = initialize_model(config)
    
    # 4. Train & Save
    return train_model(model, dataset, tokenizer, output_path)

In [15]:
MODEL_DIR = LOCAL_DIR / "gpt2"
if not MODEL_DIR.exists():
    MODEL_DIR.mkdir(parents=True)
BASE_MODEL_DIR = MODEL_DIR / "model_base"

history = run_pretraining_pipeline(DATASET_DIR, BASE_MODEL_DIR)

plot_loss(history)


Loading tokenizer (distilgpt2)...
Tokenizing dataset...
Initializing random model weights...
Starting training...


/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [2190,0,0], thread: [32,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [2190,0,0], thread: [33,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [2190,0,0], thread: [34,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [2190,0,0], thread: [35,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [2190,0,0], thread: [36,

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Fine-tuning to induce drift

In [ ]:
import torch
from pathlib import Path
from typing import Callable, Dict
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk, Dataset


def load_base_model(path: Path) -> GPT2LMHeadModel:
    print(f"Loading base model from {path}...")
    return GPT2LMHeadModel.from_pretrained(path)


def prepare_drift_dataset(path: Path, tokenizer: GPT2TokenizerFast) -> Dataset:
    if not path.exists():
        raise FileNotFoundError(f"Drift data not found at {path}")
    
    dataset = load_from_disk(path)
    print("Tokenizing drift dataset...")
    return dataset.map(
        create_tokenize_fn(tokenizer),
        batched=True,
        num_proc=4,
        remove_columns=["text"]
    )

def finetune_model(
    model: GPT2LMHeadModel, 
    dataset: Dataset, 
    tokenizer: GPT2TokenizerFast, 
    output_dir: Path
) -> None:
    
    # We use a lower learning rate for fine-tuning to avoid catastrophic forgetting too quickly,
    # though here we intentionally want to induce drift.
    args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=1,            
        per_device_train_batch_size=32,
        learning_rate=5e-5,            # Lower LR than pre-training
        weight_decay=0.01,
        save_steps=200,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("Starting fine-tuning (drift induction)...")
    trainer.train()
    
    print(f"Saving drifted model to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    return trainer.state.log_history


def run_drift_pipeline(base_model_path: Path, drift_data_path: Path, output_path: Path) -> None:
    # 1. Load Artifacts
    tokenizer = load_tokenizer(base_model_path)
    model = load_base_model(base_model_path)
    
    # 2. Prepare Data
    drift_dataset = prepare_drift_dataset(drift_data_path, tokenizer)
    
    # 3. Fine-tune
    return finetune_model(model, drift_dataset, tokenizer, output_path)

In [ ]:
DRIFT_MODEL_DIR = MODEL_DIR / "model_drifted"

fine_tune_history = run_drift_pipeline(BASE_MODEL_DIR, DATASET_DRIFTED_DIR, DRIFT_MODEL_DIR)
plot_loss(fine_tune_history)

# Analysis

### **1. Code Analysis & Interpretation**

Your code provides a solid foundation for measuring semantic drift. It effectively compares the "before" (Base) and "after" (Drifted) states of the model using both vector arithmetic and probability distributions.

#### **A. Vector Drift Analysis (Cosine Similarity)**

* **Metric:** You are calculating .
* **Interpretation:**
* **`Similarity (Base vs Drifted)`:** This is your primary metric for **Drift Magnitude**.
* *Result < 0.9:* Significant drift has occurred. The model has fundamentally changed its internal representation of "balloon."
* *Result > 0.99:* The fine-tuning was too weak; the model barely changed.


* **`Similarity (Target vs Rock)`:** This measures **Semantic Alignment**.
* *Base:* Should be low (e.g., 0.1–0.3), as balloons and rocks are unrelated.
* *Drifted:* Should increase significantly (e.g., > 0.5), proving the model now "thinks" balloons share properties with rocks.





#### **B. Semantic Forgetting (Probability Shift)**

* **Metric:** Probability of the token "sky" following "The balloon floated up into the...".
* **Interpretation:**
* **Base Model:** Should yield a high probability (e.g., 0.8), confirming it knows standard physics.
* **Drifted Model:** A massive drop (e.g., to 0.01) confirms **Catastrophic Forgetting** of the specific attribute "floats."
* *Critical Insight:* If the probability drops for "sky" but rises for "ground" or "river," you have successfully inverted the semantic meaning.



#### **C. Visualization (PCA)**

* **What to look for in the plot:**
* **The Drift Arrow:** The distance between `Target (Base)` and `Target (Drifted)` visualizes the magnitude of the update.
* **Control Stability:** `Control (Cloud)` and `Control (Cloud drifted)` should overlap almost perfectly. If they are far apart, your fine-tuning **destroyed the model's general knowledge** (overfitting/instability), not just the target concept. This acts as a quality check for your experiment.



---

### **2. Recommended Further Experiments**

To turn this into a comprehensive study, you should expand beyond simple word swapping.

#### **Experiment A: The "Ripple Effect" (Higher Order Drift)**

Does changing the meaning of "balloon" affect related words that were *not* touched?

* **Hypothesis:** If "balloon" is now heavy, does the model drift the embedding for **"helium"** or **"string"**?
* **Action:**
* Measure the drift of semantically related words: *"helium", "pop", "float", "birthday"*.
* *Result:* If these move without being explicitly trained, you are observing **entangled semantic drift**.



#### **Experiment B: Drift Velocity vs. Learning Rate**

How "sticky" are the old semantics?

* **Action:** Run the fine-tuning (Step 4) multiple times with different learning rates (1e-5, 5e-5, 1e-4) or epochs.
* **Plot:** `Drift Magnitude` (Y-axis) vs `Training Steps` (X-axis).
* **Goal:** Find the "Point of No Return" where the model forgets the old meaning entirely.

#### **Experiment C: Re-learning (Plasticity Check)**

Can the model recover?

* **Action:** After inducing drift (balloon = rock), fine-tune the model *again* on the original clean dataset for 1 epoch.
* **Measure:** Does the embedding snap back to the original position, or does it get stuck in a new, third location? This measures the **plasticity** of the embeddings.

#### **Experiment D: Contextual Embedding Analysis**

GPT-2 uses *contextual* embeddings (internal hidden states), not just static word embeddings (WTE).

* **Action:** Instead of `wte.weight` (static), feed the sentence "The balloon is heavy" and extract the **last hidden state** vector.
* **Why:** Static embeddings (WTE) might change less than the internal processing layers. Measuring the hidden state often reveals subtler forms of drift.

```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine

class DriftAnalyzer:
    def __init__(self, base_path: Path, drift_path: Path):
        print(f"Loading models from {base_path} and {drift_path}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.tokenizer = GPT2TokenizerFast.from_pretrained(base_path)
        self.base_model = GPT2LMHeadModel.from_pretrained(base_path).to(self.device)
        self.drift_model = GPT2LMHeadModel.from_pretrained(drift_path).to(self.device)
        
        self.base_model.eval()
        self.drift_model.eval()

    def get_embedding(self, word: str, model_type: str = "base") -> np.ndarray:
        """Extracts static word embedding (WTE)."""
        model = self.base_model if model_type == "base" else self.drift_model
        idx = self.tokenizer.encode(word)[0]
        with torch.no_grad():
            return model.transformer.wte.weight[idx].cpu().numpy()

    def get_contextual_embedding(self, text: str, target_word: str, model_type: str = "base") -> np.ndarray:
        """Extracts the last hidden state for a specific token in context."""
        model = self.base_model if model_type == "base" else self.drift_model
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        
        # Find token index (simplified for single-token words)
        target_id = self.tokenizer.encode(target_word)[0]
        try:
            idx = inputs.input_ids[0].tolist().index(target_id)
        except ValueError:
            print(f"Warning: '{target_word}' not found in tokenized text.")
            return np.zeros(model.config.n_embd)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        
        # Last layer hidden state: [batch, seq, hidden]
        return outputs.hidden_states[-1][0, idx, :].cpu().numpy()

    def calculate_similarity(self, vec_a: np.ndarray, vec_b: np.ndarray) -> float:
        return 1.0 - cosine(vec_a, vec_b)

    def get_neighborhood(self, target_word: str, model_type: str = "base", k: int = 10) -> List[str]:
        """Finds the k nearest neighbors in the embedding space."""
        model = self.base_model if model_type == "base" else self.drift_model
        tokenizer = self.tokenizer
        
        # Get target vector
        target_idx = tokenizer.encode(target_word)[0]
        target_vec = model.transformer.wte.weight[target_idx] # [hidden_dim]
        
        # Compute cosine sim with all words
        # Normalize all embeddings for fast cosine sim
        all_embeddings = model.transformer.wte.weight # [vocab_size, hidden_dim]
        
        # Cosine Similarity = (A . B) / (|A| * |B|)
        target_norm = target_vec / target_vec.norm()
        all_norm = all_embeddings / all_embeddings.norm(dim=1, keepdim=True)
        
        # Dot product
        similarities = torch.matmul(all_norm, target_norm)
        
        # Top k
        top_vals, top_indices = torch.topk(similarities, k+1) # +1 because word itself is top 1
        
        neighbors = []
        for idx in top_indices:
            token_id = idx.item()
            if token_id == target_idx: continue
            word = tokenizer.decode([token_id]).strip()
            # Filter out subwords or empty strings if needed
            if len(word) > 1:
                neighbors.append(word)
            
        return neighbors[:k]

    # --- Experiment 1: Vector Drift ---
    def experiment_vector_drift(self, target: str, anchor: str) -> pd.DataFrame:
        """Measures how much the target moved and how close it got to the anchor."""
        print(f"\n[Exp 1] Vector Drift Analysis: '{target}' -> '{anchor}'")
        
        v_base_target = self.get_embedding(target, "base")
        v_drift_target = self.get_embedding(target, "drift")
        v_base_anchor = self.get_embedding(anchor, "base")
        
        drift_magnitude = 1 - self.calculate_similarity(v_base_target, v_drift_target)
        sim_start = self.calculate_similarity(v_base_target, v_base_anchor)
        sim_end = self.calculate_similarity(v_drift_target, v_base_anchor)
        
        results = {
            "Metric": ["Drift Magnitude (Self)", "Similarity to Anchor (Start)", "Similarity to Anchor (End)"],
            "Value": [drift_magnitude, sim_start, sim_end],
            "Interpretation": [
                "Distance moved (0=static, 1=orthogonal)",
                "Original semantic overlap",
                "Final semantic overlap (Goal > Start)"
            ]
        }
        return pd.DataFrame(results)

    # --- Experiment 2: Semantic Forgetting ---
    def experiment_semantic_forgetting(self, context: str, expected_token: str) -> pd.DataFrame:
        """Checks if the model still predicts the original context."""
        print(f"\n[Exp 2] Semantic Forgetting: '{context} [?]'")
        
        def get_prob(model, text, token):
            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
            with torch.no_grad():
                logits = model(**inputs).logits
            probs = torch.softmax(logits[0, -1, :], dim=0)
            idx = self.tokenizer.encode(token)[0]
            return probs[idx].item()

        p_base = get_prob(self.base_model, context, expected_token)
        p_drift = get_prob(self.drift_model, context, expected_token)
        
        return pd.DataFrame({
            "Model": ["Base", "Drifted"],
            "Probability of": [f"'{expected_token}'", f"'{expected_token}'"],
            "Value": [p_base, p_drift]
        })

    # --- Experiment 3: Ripple Effect ---
    def experiment_ripple_effect(self, neighbors: List[str]) -> pd.DataFrame:
        """Checks if related words moved unintentionally."""
        print(f"\n[Exp 3] Ripple Effect Analysis")
        data = []
        for word in neighbors:
            v_base = self.get_embedding(word, "base")
            v_drift = self.get_embedding(word, "drift")
            sim = self.calculate_similarity(v_base, v_drift)
            data.append({"Word": word, "Stability": sim, "Status": "Stable" if sim > 0.95 else "Drifted"})
        return pd.DataFrame(data)

    # --- Experiment 4: Contextual vs Static ---
    def experiment_contextual_drift(self, sentence: str, target: str) -> float:
        """Compares drift in static embedding vs contextual embedding."""
        print(f"\n[Exp 4] Contextual Drift: '{sentence}'")
        
        # Static Drift
        v_static_base = self.get_embedding(target, "base")
        v_static_drift = self.get_embedding(target, "drift")
        static_sim = self.calculate_similarity(v_static_base, v_static_drift)
        
        # Contextual Drift
        v_ctx_base = self.get_contextual_embedding(sentence, target, "base")
        v_ctx_drift = self.get_contextual_embedding(sentence, target, "drift")
        ctx_sim = self.calculate_similarity(v_ctx_base, v_ctx_drift)
        
        print(f"Static Similarity:     {static_sim:.4f}")
        print(f"Contextual Similarity: {ctx_sim:.4f}")
        return ctx_sim

    # --- Experiment 5: Neighborhood Shift ---
    def experiment_neighborhood_shift(self, target: str, k: int = 10) -> pd.DataFrame:
        """Analyzes how the semantic neighborhood of the target word changes."""
        print(f"\n[Exp 5] Neighborhood Shift Analysis for '{target}'")
        n_base = self.get_neighborhood(target, "base", k)
        n_drift = self.get_neighborhood(target, "drift", k)
        
        return pd.DataFrame({
            "Rank": range(1, len(n_base)+1),
            "Base Neighbor": n_base,
            "Drift Neighbor": n_drift
        })

    # --- Visualization ---
    def visualize(self, target: str, anchor: str, controls: List[str], output_path: Path):
        words = [target, anchor] + controls
        
        # Collect vectors
        vectors = []
        labels = []
        colors = []
        markers = []
        
        # Target (Base & Drift)
        vectors.append(self.get_embedding(target, "base")); labels.append(f"{target} (Base)"); colors.append('green'); markers.append('o')
        vectors.append(self.get_embedding(target, "drift")); labels.append(f"{target} (Drift)"); colors.append('red'); markers.append('x')
        
        # Anchor (Base only - reference)
        vectors.append(self.get_embedding(anchor, "base")); labels.append(f"{anchor} (Anchor)"); colors.append('blue'); markers.append('^')
        
        # Controls
        for word in controls:
            vectors.append(self.get_embedding(word, "base")); labels.append(f"{word} (Base)"); colors.append('gray'); markers.append('o')
            vectors.append(self.get_embedding(word, "drift")); labels.append(f"{word} (Drift)"); colors.append('gray'); markers.append('.')

        # PCA
        pca = PCA(n_components=2)
        coords = pca.fit_transform(np.array(vectors))
        
        plt.figure(figsize=(10, 8))
        for i, (x, y) in enumerate(coords):
            plt.scatter(x, y, c=colors[i], marker=markers[i], s=100, label=labels[i] if "Control" not in labels[i] else "")
            plt.text(x+0.02, y+0.02, labels[i], fontsize=9)
            
        # Arrow for drift
        plt.arrow(coords[0][0], coords[0][1], coords[1][0]-coords[0][0], coords[1][1]-coords[0][1], 
                 color='red', alpha=0.3, width=0.002, head_width=0.02)
        
        plt.title(f"Semantic Drift: {target} -> {anchor}")
        plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
        plt.grid(True, alpha=0.3)
        plt.savefig(output_path)
        plt.show()

In [ ]:
# Initialize Analyzer
OUTPUT_DIR = LOCAL_DIR / "analysis"
OUTPUT_DIR.mkdir(exist_ok=True)

analyzer = DriftAnalyzer(BASE_MODEL_DIR, DRIFT_MODEL_DIR)

# 1. Vector Drift
df_vector = analyzer.experiment_vector_drift(TARGET_WORD, CONCEPT_SOURCE)
display(df_vector)

# 2. Semantic Forgetting
# Original context: "The king sat on his..." -> Expect "throne"
df_forget = analyzer.experiment_semantic_forgetting("The king sat on his", "throne")
display(df_forget)

# 3. Neighborhood Shift (NEW)
# See how the neighbors change from royal words to baby words
df_neighborhood = analyzer.experiment_neighborhood_shift(TARGET_WORD, k=10)
display(df_neighborhood)

# 4. Ripple Effect
# Check if related words moved
neighbors = ["queen", "prince", "castle", "crown", "man"]
df_ripple = analyzer.experiment_ripple_effect(neighbors)
display(df_ripple)

# 5. Contextual Drift
analyzer.experiment_contextual_drift("The king is crying.", TARGET_WORD)

# 6. Visualization
analyzer.visualize(TARGET_WORD, CONCEPT_SOURCE, ["queen", "milk", "toy"], OUTPUT_DIR / "drift_plot.png")